# Manhwa Studio — Renderizador GPU no Google Colab
**Antes de executar:** no menu do Colab, selecione **Ambiente de execução → Alterar tipo de ambiente de execução → Acelerador de hardware → GPU (T4, se disponível)**. Salve a configuração, conecte o ambiente e execute todas as células em ordem.

A API usa NVIDIA NVENC (`h264_nvenc`) quando uma GPU NVIDIA e o encoder estão disponíveis. Se não estiverem, usa FFmpeg CPU como fallback e informa isso no teste do site. URL HTTPS e token são gerados novamente a cada sessão. Mantenha esta aba do Colab aberta durante a renderização.

In [ ]:
# 1) Instalar FFmpeg e dependências
!apt-get -qq update
!apt-get -qq install -y ffmpeg curl
!pip -q install fastapi uvicorn python-multipart requests
!ffmpeg -version | head -n 1
import subprocess, shutil
print('GPU NVIDIA encontrada:', bool(shutil.which('nvidia-smi')))
if shutil.which('nvidia-smi'):
    subprocess.run(['nvidia-smi'])
encoders = subprocess.run(['ffmpeg', '-hide_banner', '-encoders'], capture_output=True, text=True).stdout
print('Encoder NVENC disponível:', 'h264_nvenc' in encoders)


In [ ]:
# 2) Instalar Cloudflare Tunnel (URL HTTPS temporária)
!curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version


In [ ]:
# 3) Baixar a API GPU do GitHub e validar a sintaxe
!curl -fsSL https://raw.githubusercontent.com/manhwavaz1-lab/manhwa-recreator-studio/main/colab/manhwa_cpu_api.py -o /content/manhwa_cpu_api.py
!python -m py_compile /content/manhwa_cpu_api.py
print('API baixada e validada.')


In [ ]:
# 4) Iniciar API e túnel. Mantenha esta célula em execução.
import os, re, secrets, subprocess, time, threading, queue, requests
from IPython.display import display, Markdown

# Encerrar processos antigos caso esta célula seja executada novamente.
for process_name in ('api_process', 'tunnel_process'):
    old = globals().get(process_name)
    if old is not None and old.poll() is None:
        old.terminate()

RENDER_TOKEN = secrets.token_urlsafe(32)
os.environ['MANHWA_RENDER_TOKEN'] = RENDER_TOKEN
api_process = subprocess.Popen(
    ['uvicorn', 'manhwa_cpu_api:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd='/content', stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT
)

# Aguardar a API ficar pronta antes de abrir o túnel.
api_ready = False
for _ in range(60):
    if api_process.poll() is not None:
        raise RuntimeError('A API encerrou durante a inicialização. Reexecute a célula da API e confira as dependências.')
    try:
        check = requests.get('http://127.0.0.1:8000/health', timeout=2)
        if check.ok and check.json().get('status') == 'ok':
            api_ready = True
            break
    except requests.RequestException:
        time.sleep(1)
if not api_ready:
    api_process.terminate()
    raise RuntimeError('A API não respondeu em 60 segundos.')

log_queue = queue.Queue()
tunnel_process = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
def collect_tunnel_logs():
    for line in iter(tunnel_process.stdout.readline, ''):
        log_queue.put(line)
reader_thread = threading.Thread(target=collect_tunnel_logs, daemon=True)
reader_thread.start()

PUBLIC_URL = None
deadline = time.time() + 90
while time.time() < deadline and tunnel_process.poll() is None:
    try:
        line = log_queue.get(timeout=1)
    except queue.Empty:
        continue
    found = re.search(r'https://[a-z0-9-]+\\.trycloudflare\\.com', line)
    if found:
        PUBLIC_URL = found.group(0)
        break
if not PUBLIC_URL:
    tunnel_process.terminate()
    api_process.terminate()
    raise RuntimeError('O Cloudflare Tunnel não forneceu uma URL em 90 segundos. Execute novamente esta célula.')

display(Markdown(
    '## Renderizador conectado\n\n'
    + '**URL HTTPS:** `' + PUBLIC_URL + '`\n\n'
    + '**Token desta sessão (copie completo):** `' + RENDER_TOKEN + '`\n\n'
    + '**Teste:** `' + PUBLIC_URL + '/health`\n\n'
    + 'Cole a URL e o token no site e clique em **Testar conexão**. Não compartilhe o token.'
))
print('API e túnel ativos. Deixe o Colab aberto enquanto o site renderiza. O endpoint /health também exige o token; no site, cole URL e token e clique em Testar conexão.')


## Como usar no site
1. Selecione GPU no tipo de ambiente de execução antes de iniciar o notebook.
2. Execute todas as células em ordem.
3. Na última célula, copie a URL HTTPS completa e o token completo.
4. No site, cole a URL no campo URL e o token no campo Token. Não copie os rótulos, aspas, crases ou espaços extras.
5. Clique em **Testar conexão**. O site agora envia o token no teste; token incorreto ou URL de sessão antiga retorna erro 401.
6. Quando o status indicar GPU/NVENC, inicie a renderização. Se o Colab não disponibilizar GPU/NVENC, o servidor informa fallback para CPU.

**Importante:** URL e token mudam a cada sessão. Se reiniciar ou reconectar o Colab, execute novamente a célula de inicialização e substitua os dois campos no site. Não compartilhe o token.

In [ ]:
# Opcional: executar para encerrar API e túnel
for process in (globals().get('tunnel_process'), globals().get('api_process')):
    if process is not None and process.poll() is None:
        process.terminate()
print('API e túnel encerrados.')
